In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2


In [ ]:
from tqdm import tqdm

In [ ]:
# Download training data from open datasets.
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.
    float32, scale=True)]),
)

# Download test data from open datasets.
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

100%|██████████| 26.4M/26.4M [00:02<00:00, 11.5MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 172kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.23MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 9.94MB/s]


In [ ]:
batch_size = 64

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


In [ ]:
class RandomNet(nn.Module):
    def __init__(self, hidden_size_1=1000, hidden_size_2=2000):
        super(RandomNet,self).__init__()
        self.linear1 = nn.Linear(28*28, hidden_size_1)
        self.linear2 = nn.Linear(hidden_size_1, hidden_size_2)
        self.linear3 = nn.Linear(hidden_size_2, 10)
        self.relu = nn.ReLU()

    def forward(self, img):
        x = img.view(-1, 28*28)
        x = self.relu(self.linear1(x))
        x = self.relu(self.linear2(x))
        x = self.linear3(x)
        return x

net = RandomNet()

In [ ]:
def train(train_loader, net, epochs=5, total_iterations_limit=None):
    cross_el = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=0.001)

    total_iterations = 0

    for epoch in range(epochs):
        net.train()

        loss_sum = 0
        num_iterations = 0

        data_iterator = tqdm(train_loader, desc=f'Epoch {epoch+1}')
        if total_iterations_limit is not None:
            data_iterator.total = total_iterations_limit
        for data in data_iterator:
            num_iterations += 1
            total_iterations += 1
            x, y = data
            optimizer.zero_grad()
            output = net(x.view(-1, 28*28))
            loss = cross_el(output, y)
            loss_sum += loss.item()
            avg_loss = loss_sum / num_iterations
            data_iterator.set_postfix(loss=avg_loss)
            loss.backward()
            optimizer.step()

            if total_iterations_limit is not None and total_iterations >= total_iterations_limit:
                return

train(train_dataloader, net, epochs=1)

Epoch 1: 100%|██████████| 938/938 [01:11<00:00, 13.08it/s, loss=0.477]


In [ ]:
# First, let's fix the class name typo (nn.module should be nn.Module) and instantiate the class
class LoRA(nn.Module):
  def __init__(self, features_in, features_out,alpha=1, rank=4):
    super().__init__()
    self.loraA = nn.Parameter(torch.zeros((rank, features_out)))
    self.loraB = nn.Parameter(torch.zeros((features_in, rank)))
    nn.init.normal_(self.loraA)
    self.scale = alpha/rank
    self.enable = True
  def forward(self, original_weights):
    if self.enable:
      return original_weights + torch.matmul(self.loraB,self.loraA)*self.scale
    else:
      return original_weights

# Instantiate the module
lora_layer = LoRA(features_in=28, features_out=10, rank=4)

# Print the loraA parameter
print("loraA parameter:")
print(lora_layer.loraA)

loraA parameter:
Parameter containing:
tensor([[ 0.0649,  1.1239,  0.7427,  2.4503, -0.2406,  2.4499, -0.8224,  1.0729,
          0.8235,  2.0251],
        [ 1.1265, -0.9867,  0.5741, -0.2044, -0.9676, -0.4634, -0.2779,  1.8934,
         -0.5984, -0.2686],
        [-0.8949, -0.1835, -0.7745, -0.0476, -1.4905,  1.2079,  1.6770, -0.0920,
         -0.7144, -0.2627],
        [-2.2459, -1.1460,  0.6624,  0.0989, -0.9553,  0.3356,  0.0875,  1.8676,
          1.0610,  0.2316]], requires_grad=True)


In [ ]:
def test():
  correct=0
  total=0
  wrong_counts = [0 for i in range(10)]
  with torch.no_grad():
    for data in test_dataloader:
      x,y = data
      output = net(x.view(-1,28*28))
      for idx, i in enumerate(output):
                total += 1  # Increment total for every sample
                if torch.argmax(i) == y[idx]:
                    correct +=1
                else:
                    wrong_counts[y[idx]] +=1

  print(f'Accuracy: {round(correct/total, 3)}')
  for i in range(len(wrong_counts)):
        print(f'wrong counts for the digit {i}: {wrong_counts[i]}')

test()

Accuracy: 0.86
wrong counts for the digit 0: 168
wrong counts for the digit 1: 28
wrong counts for the digit 2: 242
wrong counts for the digit 3: 180
wrong counts for the digit 4: 145
wrong counts for the digit 5: 57
wrong counts for the digit 6: 431
wrong counts for the digit 7: 37
wrong counts for the digit 8: 29
wrong counts for the digit 9: 81


In [ ]:
import torch.nn.utils.parametrize as parametrize

def linear_layer_parameterization(layer, rank=1, lora_alpha=1):
    # Only add the parameterization to the weight matrix, ignore the Bias

    # From section 4.2 of the paper:
    #   We limit our study to only adapting the attention weights for downstream tasks and freeze the MLP modules (so they are not trained in downstream tasks) both for simplicity and parameter-efficiency.
    #   [...]
    #   We leave the empirical investigation of [...], and biases to a future work.

    features_in, features_out = layer.weight.shape
    return LoRA(
        features_in, features_out, alpha=lora_alpha,rank=rank
    )

parametrize.register_parametrization(
    net.linear1, "weight", linear_layer_parameterization(net.linear1)
)
parametrize.register_parametrization(
    net.linear2, "weight", linear_layer_parameterization(net.linear2)
)
parametrize.register_parametrization(
    net.linear3, "weight", linear_layer_parameterization(net.linear3)
)


def enable_disable_lora(enabled=True):
    for layer in [net.linear1, net.linear2, net.linear3]:
        layer.parametrizations["weight"][0].enable = enabled

In [ ]:
# 1. Freeze all original weights (including the original weights under PyTorch parametrizations)
for name, param in net.named_parameters():
    if 'loraA' in name or 'loraB' in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

# 2. Verify which parameters are trainable
print("Trainable Parameters:")
for name, param in net.named_parameters():
    if param.requires_grad:
        print(f"{name}: {param.shape}")

Trainable Parameters:
linear1.parametrizations.weight.0.loraA: torch.Size([1, 784])
linear1.parametrizations.weight.0.loraB: torch.Size([1000, 1])
linear2.parametrizations.weight.0.loraA: torch.Size([1, 1000])
linear2.parametrizations.weight.0.loraB: torch.Size([2000, 1])
linear3.parametrizations.weight.0.loraA: torch.Size([1, 2000])
linear3.parametrizations.weight.0.loraB: torch.Size([10, 1])


In [ ]:
# 3. Retrain the model with only LoRA weights being optimized
train(train_dataloader, net, epochs=1)

Epoch 1: 100%|██████████| 938/938 [01:12<00:00, 12.86it/s, loss=0.278]


In [ ]:
# 4. Test the model's updated performance
enable_disable_lora(True)
test()

Accuracy: 0.879
wrong counts for the digit 0: 136
wrong counts for the digit 1: 34
wrong counts for the digit 2: 235
wrong counts for the digit 3: 105
wrong counts for the digit 4: 181
wrong counts for the digit 5: 41
wrong counts for the digit 6: 355
wrong counts for the digit 7: 59
wrong counts for the digit 8: 23
wrong counts for the digit 9: 41


In [ ]:
enable_disable_lora(False)
test()

Accuracy: 0.864
wrong counts for the digit 0: 158
wrong counts for the digit 1: 42
wrong counts for the digit 2: 313
wrong counts for the digit 3: 177
wrong counts for the digit 4: 106
wrong counts for the digit 5: 56
wrong counts for the digit 6: 391
wrong counts for the digit 7: 50
wrong counts for the digit 8: 21
wrong counts for the digit 9: 45
